In [134]:
import pandas as pd
from scipy.spatial.distance import pdist, cdist
import numpy as np
import math
import datetime as dt
from datetime import timedelta
import itertools
from collections import defaultdict
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../tests")))

In [135]:
# sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import filters
import daphmeIO as loader
import constants as constants
import stop_detection_modified as SD
import stop_detection as baseSD

In [136]:
path = '../data/sample4/'

traj_cols = {'user_id':'user_id',
             'latitude':'dev_lat',
             'longitude':'dev_lon',
             'datetime':'local_datetime'}

df = loader.from_file(path, traj_cols=traj_cols, format='csv')

In [84]:
df = filters.to_projection(df, latitude='dev_lat', longitude='dev_lon')

In [66]:
df['timestamp'] = df['local_datetime'].astype('int64') // 10**9

In [137]:
df

,user_id,dev_lat,dev_lon,local_datetime
0,wizardly_joliot,38.321711,-36.667334,2024-01-01 14:29:00
1,wizardly_joliot,38.321676,-36.667365,2024-01-01 14:35:00
2,wonderful_swirles,38.321017,-36.667869,2024-01-01 15:06:00
3,youthful_galileo,38.321625,-36.666612,2024-01-01 08:47:00
4,youthful_galileo,38.321681,-36.666841,2024-01-01 09:59:00
...,...,...,...,...
25830,angry_spence,38.320399,-36.667438,2024-01-15 07:23:00
25831,angry_spence,38.320413,-36.667469,2024-01-15 07:29:00
25832,angry_spence,38.320384,-36.667455,2024-01-15 07:33:00
25833,angry_spence,38.320349,-36.667473,2024-01-15 07:39:00


In [110]:
sample_user_df = df[df['user_id'] == "angry_spence"]

In [166]:
time_thresh = 100
dist_thresh = 40
min_pts = 10

In [88]:
stops = SD.temporal_dbscan(sample_user_df, time_thresh, dist_thresh, min_pts, traj_cols=traj_cols)

In [89]:
stops.value_counts()

cluster  core
-1       -1      1696
Name: count, dtype: int64

In [139]:
sample_user_df = df[df['user_id'] == "angry_spence"]

In [78]:
df = filters.to_projection(df, latitude='dev_lat', longitude='dev_lon')
df['unix_timestamp'] = df['local_datetime'].astype('int64') // 10**9

In [113]:
sample_user_df = sample_user_df.set_index(traj_cols['datetime'], drop=False)

In [162]:
def extract_middle(data):
    current = data.iloc[0]['cluster']
    x = (data.cluster != current).values
    if len(np.where(x)[0]) == 0:  # There is no inbetween
        return (len(data), len(data))
    else:
        i = np.where(x)[0][
            0]  # First index where the cluster is not the value of the first entry's cluster
    if len(np.where(~x[i:])[
               0]) == 0:  # There is no current again (i.e., the first cluster does not reappear, so the middle is actually the tail)
        return (i, len(data))
    else:  # Current reappears
        j = i + np.where(~x[i:])[0][0]
    return (i, j)

def haversine_distance(coord1, coord2):
    earth_radius_meters = 6371000  # Earth's radius in meters
    delta_lat = coord2[0] - coord1[0]
    delta_lon = coord2[1] - coord1[1]
    a = np.sin(delta_lat / 2.0) ** 2 + np.cos(coord1[0]) * np.cos(
        coord2[0]) * np.sin(delta_lon / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return earth_radius_meters * c  # Distance in meters

In [236]:
# this works
def find_neighbors(data, time_thresh, dist_thresh, long_lat, datetime, traj_cols):
    if long_lat:
        coords = np.radians(data[[traj_cols['latitude'], traj_cols['longitude']]].values)
    else:
        coords = data[[traj_cols['x'], traj_cols['y']]].values

    if datetime:
        times = data[traj_cols['datetime']].astype('datetime64[s]').astype(int).values
    else:
        first_timestamp = data[traj_cols['timestamp']].iloc[0]
        timestamp_length = len(str(first_timestamp))
        
        if timestamp_length > 10:
            if timestamp_length == 13:
                times = data[traj_cols['timestamp']].values.view('int64') // 10 ** 3
            elif timestamp_length == 19:
                times = data[traj_cols['timestamp']].values.view('int64') // 10 ** 9   
        else:
            times = data[traj_cols['timestamp']].values

    # Pairwise time differences
    time_diffs = np.abs(times[:, np.newaxis] - times)
    time_diffs = time_diffs.astype(int)

    # Time threshold calculation using broadcasting
    within_time_thresh = np.triu(time_diffs <= (time_thresh * 60), k=1)
    time_pairs = np.where(within_time_thresh)
    
    # Distance calculation
    if long_lat:
        distances = np.array([haversine_distance(coords[i], coords[j]) for i, j in zip(*time_pairs)])
    else:
        distances_sq = (coords[time_pairs[0], 0] - coords[time_pairs[1], 0])**2 + \
                       (coords[time_pairs[0], 1] - coords[time_pairs[1], 1])**2
        distances = np.sqrt(distances_sq)

    # Filter by distance threshold
    neighbor_pairs = distances < dist_thresh
    
    # Building the neighbor dictionary
    neighbor_dict = defaultdict(set)
    for i, j in zip(time_pairs[0][neighbor_pairs], time_pairs[1][neighbor_pairs]):
        neighbor_dict[times[i]].add(times[j])
        neighbor_dict[times[j]].add(times[i])

    return neighbor_dict

In [246]:
# this works
def dbscan(data, time_thresh, dist_thresh, min_pts, long_lat, datetime, traj_cols, neighbor_dict=None):
    if datetime:
        valid_times = data[traj_cols['datetime']].astype('datetime64[s]').astype(int).values
        original_times = data[traj_cols['datetime']].values
    else:
        first_timestamp = data[traj_cols['timestamp']].iloc[0]
        timestamp_length = len(str(first_timestamp))
        
        if timestamp_length > 10:
            if timestamp_length == 13:
                warnings.warn(
                    f"The '{data[traj_cols['timestamp']]}' column appears to be in milliseconds. "
                    "This may lead to inconsistencies."
                )
                valid_times = data[traj_cols['timestamp']].values.view('int64') // 10 ** 3
                original_times = data[traj_cols['timestamp']].values
            elif timestamp_length == 19:
                warnings.warn(
                    f"The '{timestamp_col_name}' column appears to be in nanoseconds. "
                    "This may lead to inconsistencies."
                )
                valid_times = data[traj_cols['timestamp']].values.view('int64') // 10 ** 9 
                original_times = data[traj_cols['timestamp']].values
        else:
            valid_times = data[traj_cols['timestamp']].values
            original_times = data[traj_cols['timestamp']].values
    
    if not neighbor_dict:
        neighbor_dict = find_neighbors(data, time_thresh, dist_thresh, long_lat, datetime, traj_cols)
    else:
        neighbor_dict = defaultdict(set,
                                    {k: v.intersection(valid_times) for k, v in
                                     neighbor_dict.items() if
                                     k in valid_times})
    
    cluster_df = pd.Series(-2, index=valid_times, name='cluster')
    core_df = pd.Series(-1, index=valid_times, name='core')

    cid = -1  # Initialize cluster label

    for i, cluster in cluster_df.items():
        if cluster < 0:  # Check if point is not yet in a cluster
            if len(neighbor_dict[i]) < min_pts:
                cluster_df[i] = -1  # Mark as noise if below min_pts
            else:
                cid += 1
                cluster_df[i] = cid  # Assign new cluster label
                core_df[i] = cid  # Assign new core label
                S = list(neighbor_dict[i])  # Initialize stack with neighbors

                while S:
                    j = S.pop()
                    if cluster_df[j] < 0:  # Process if not yet in a cluster
                        cluster_df[j] = cid
                        if len(neighbor_dict[j]) >= min_pts:
                            core_df[j] = cid  # Assign core label
                            for k in neighbor_dict[j]:
                                if cluster_df[k] < 0:
                                    S.append(k)  # Add new neighbors

    return pd.DataFrame({'cluster': cluster_df, 'core': core_df})

In [238]:
def process_clusters(data, time_thresh, dist_thresh, min_pts, output, long_lat, datetime, traj_cols, 
                     cluster_df=None, neighbor_dict=None, min_duration=4):

    if not neighbor_dict:
        neighbor_dict = find_neighbors(data, time_thresh, dist_thresh, long_lat, datetime, traj_cols)
    if cluster_df is None:
        cluster_df = dbscan(data, time_thresh, dist_thresh, min_pts, long_lat, datetime, traj_cols, neighbor_dict=neighbor_dict)
    if len(cluster_df) < min_pts:
        return False

    cluster_df = cluster_df[cluster_df['cluster'] != -1]  # Remove noise pings

    # All pings are in the same cluster
    if len(cluster_df['cluster'].unique()) == 1:
        # We rerun dbscan because possibly these points no longer hold their own
        x = dbscan(data = data.loc[cluster_df.index], time_thresh = time_thresh, dist_thresh = dist_thresh,
                   min_pts = min_pts, long_lat = long_lat, datetime = datetime, traj_cols = traj_cols, neighbor_dict = neighbor_dict)
        
        y = x.loc[x['cluster'] != -1]
        z = x.loc[x['core'] != -1]

        if len(y) > 0:
            duration = int((y.index.max() - y.index.min()) // 60)

            if duration > min_duration:
                cid = max(output['cluster']) + 1 # Create new cluster id
                output.loc[y.index, 'cluster'] = cid
                output.loc[z.index, 'core'] = cid
            
            return True
        elif len(y) == 0: # The points in df, despite originally being part of a cluster, no longer hold their own
            return False

    # There are no clusters
    elif len(cluster_df['cluster'].unique()) == 0:
        return False
   
    # There is more than one cluster
    elif len(cluster_df['cluster'].unique()) > 1:
        i, j = extract_middle(cluster_df)  # Indices of the "middle" of the cluster
        
        # Recursively processes clusters
        if process_clusters(data, time_thresh, dist_thresh, min_pts, output, long_lat, datetime, traj_cols, cluster_df = cluster_df[i:j]):  # Valid cluster in the middle
            process_clusters(data, time_thresh, dist_thresh, min_pts, output,
                             long_lat, datetime, traj_cols, cluster_df = cluster_df[:i])  # Process the initial stub
            process_clusters(data, time_thresh, dist_thresh, min_pts, output,
                             long_lat, datetime, traj_cols, cluster_df = cluster_df[j:])  # Process the "tail"
            return True
        else:  # No valid cluster in the middle
            return process_clusters(data, time_thresh, dist_thresh, min_pts, output, long_lat, datetime, traj_cols, pd.concat([cluster_df[:i], cluster_df[j:]]))

In [239]:
def temporal_dbscan(data, time_thresh, dist_thresh, min_pts, complete_output=False, traj_cols=None, **kwargs):
    
    # Check if user wants long and lat
    long_lat = 'latitude' in kwargs and 'longitude' in kwargs and kwargs[
        'latitude'] in data.columns and kwargs['longitude'] in data.columns

    # Check if user wants datetime
    datetime = 'datetime' in kwargs and kwargs['datetime'] in data.column

    # Set initial schema
    if not traj_cols:
        traj_cols = {}

    traj_cols = loader._update_schema(traj_cols, kwargs)
    traj_cols = loader._update_schema(constants.DEFAULT_SCHEMA, traj_cols)

    # Tests to check for spatial and temporal columns
    loader._has_spatial_cols(data.columns, traj_cols)
    loader._has_time_cols(data.columns, traj_cols)

    # Setting x and y as defaults if not specified by user in either traj_cols or kwargs
    if traj_cols['x'] in data.columns and traj_cols['y'] in data.columns:
        long_lat = False
    else:
        long_lat = True

    # Setting timestamp as default if not specified by user in either traj_cols or kwargs
    if traj_cols['timestamp'] in data.columns:
        datetime = False
    else:
        datetime = True

    if datetime:
        valid_times = data[traj_cols['datetime']].astype('datetime64[s]').astype(int).values
    else:
        first_timestamp = data[traj_cols['timestamp']].iloc[0]
        timestamp_length = len(str(first_timestamp))
        
        if timestamp_length > 10:
            if timestamp_length == 13:
                warnings.warn(
                    f"The '{data[traj_cols['timestamp']]}' column appears to be in milliseconds. "
                    "This may lead to inconsistencies."
                )
                valid_times = data[traj_cols['timestamp']].values.view('int64') // 10 ** 3
            elif timestamp_length == 19:
                warnings.warn(
                    f"The '{timestamp_col_name}' column appears to be in nanoseconds. "
                    "This may lead to inconsistencies."
                )
                valid_times = data[traj_cols['timestamp']].values.view('int64') // 10 ** 9 
        else:
            valid_times = data[traj_cols['timestamp']].values

    data_temp = data.copy()
    data_temp.index = valid_times        

    output = pd.DataFrame({'cluster': -1, 'core': -1}, index=valid_times)

    process_clusters(data_temp, time_thresh, dist_thresh, min_pts, output, long_lat, datetime, traj_cols, min_duration=4)

    output.index = list(data['local_datetime'])
    
    return output

In [243]:
output = dbscan(sample_user_df, time_thresh, dist_thresh, min_pts, long_lat = True, datetime = True, traj_cols = traj_cols, neighbor_dict=None)
output

,cluster,core
1704104460,-1,-3
1704104820,-1,-3
1704104940,-1,-3
1704105540,-1,-3
1704105720,-1,-3
...,...,...
1705303380,49,49
1705303740,49,49
1705303980,49,49
1705304340,49,49


In [244]:
temporal_output = temporal_dbscan(sample_user_df, time_thresh, dist_thresh, min_pts, complete_output=False, traj_cols=traj_cols)
temporal_output

,cluster,core
2024-01-01 10:21:00,-1,-1
2024-01-01 10:27:00,-1,-1
2024-01-01 10:29:00,-1,-1
2024-01-01 10:39:00,-1,-1
2024-01-01 10:42:00,-1,-1
...,...,...
2024-01-15 07:23:00,1,1
2024-01-15 07:29:00,1,1
2024-01-15 07:33:00,1,1
2024-01-15 07:39:00,1,1
